# 04 — Analysis

Every number in the paper, computed from the stored responses in `results/`. No model is queried.

Run order: **04 → 05 → 06 → 07**. This notebook also writes Supplementary Tables S1, S3, S4, S5,
`supplementary/supplementary_numbers.json` and `data/processed/providers_observed.csv`.

In [1]:
# ============================================================================
# 04a — SETUP
# ============================================================================
import json, hashlib
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import binomtest, spearmanr

ROOT = Path.cwd().parent
PROC, RES, SUPP = ROOT/"data"/"processed", ROOT/"results", ROOT/"supplementary"
(SUPP/"tables").mkdir(parents=True, exist_ok=True)
ONESIDED_CUTOFF = 0.85    # one label on >85% of committed answers -> gap uninterpretable
N_BOOT, SEED = 4000, 1    # bootstrap over proteins
rng = np.random.default_rng(SEED)

A_all = pd.concat([pd.read_json(p, lines=True) for p in sorted(RES.glob("*.jsonl"))],
                  ignore_index=True)
A = A_all[A_all.http==200].copy()
A["abstained"] = A.pred=="not_established"
A["committed"] = A.pred.isin(["stimulates","inhibits"])
A["correct"]   = A.pred.notna() & (A.pred==A.truth)

MAIN = A[(A.arm=="main") & ~A.is_decoy]
DEC  = A[(A.arm=="main") &  A.is_decoy]
SW, WEB, U = A[A.arm=="swap"], A[A.arm=="web"], A[A.arm=="usually"]
MODELS_ = sorted(MAIN.model.unique())
PROTS   = sorted(MAIN.source.unique())

def acc(d, c): return d[d.condition==c].correct.mean()*100
def gap(d, m=None):
    if m is not None: d = d[d.model==m]
    return acc(d,"habit_agrees") - acc(d,"habit_misleads")
def paired(d, value="correct"):
    """One row per protein, one column per condition."""
    return (d.pivot_table(index="source", columns="condition", values=value, aggfunc="first")
             .reindex(PROTS))
def mcnemar(p):
    p = p.dropna().astype(bool)
    ao = int((p.habit_agrees & ~p.habit_misleads).sum())
    bo = int((~p.habit_agrees & p.habit_misleads).sum())
    return ao, bo, (binomtest(ao, ao+bo, .5).pvalue if ao+bo else 1.0)
def holm(ps):
    ps = np.asarray(ps, float); out = np.empty_like(ps); run = 0.0
    for r, i in enumerate(np.argsort(ps)):
        run = max(run, (len(ps)-r)*ps[i]); out[i] = min(1.0, run)
    return out
def boot_idx(n=N_BOOT): return rng.integers(0, len(PROTS), (n, len(PROTS)))
def gap_from(p, idx):
    a, b = p.habit_agrees.values.astype(float), p.habit_misleads.values.astype(float)
    return (np.nanmean(a[idx], 1) - np.nanmean(b[idx], 1))*100

skew = {m: MAIN[(MAIN.model==m) & MAIN.committed].pred.value_counts(normalize=True).max()
        for m in MODELS_}
ONESIDED = [m for m in MODELS_ if skew[m] > ONESIDED_CUTOFF]
INFORM   = [m for m in MODELS_ if m not in ONESIDED]
R = {m: {} for m in MODELS_}          # per-model results, become Table S1

print(f"{len(A):,} scored responses | {len(MODELS_)} models | {MAIN.item_id.nunique()} questions "
      f"| {DEC.item_id.nunique()} fabricated pairs | web arm: {sorted(WEB.model.unique())}")
print(f"one-sided (> {ONESIDED_CUTOFF:.0%} one label): "
      f"{ {m: round(float(skew[m])*100,1) for m in ONESIDED} }")
nxt = max(v for m, v in skew.items() if m not in ONESIDED)
print(f"  next most one-sided model: {nxt:.1%} -> any cutoff between {nxt:.0%} and "
      f"{min(skew[m] for m in ONESIDED):.0%} excludes the same models")

10,988 scored responses | 16 models | 216 questions | 120 fabricated pairs | web arm: ['gemini-3.7-flash', 'gpt-5.6-terra']
one-sided (> 85% one label): {'llama-3-70b': 89.3, 'llama-3.1-8b': 97.6}
  next most one-sided model: 75.3% -> any cutoff between 75% and 89% excludes the same models


In [2]:
# ============================================================================
# 04b — THE GAP: all responses, committed only, uncertainty, chance
# ============================================================================
print(f"{'model':<19}{'agree':>7}{'mislead':>8}{'gap':>7}{'95% CI':>15}  |"
      f"{'agree*':>7}{'mislead*':>9}{'gap*':>7}  {'exc* vs 50%':>11}")
print("-"*100)
for m in MODELS_:
    g = MAIN[MAIN.model==m]; c = g[g.committed]; p = paired(g)
    bs = gap_from(p, boot_idx())
    ce = c[c.condition=="habit_misleads"]; ch = c[c.condition=="habit_agrees"]
    pe_lo = binomtest(int(ce.correct.sum()), len(ce), .5, alternative="less").pvalue
    pe_hi = binomtest(int(ce.correct.sum()), len(ce), .5, alternative="greater").pvalue
    ph_hi = binomtest(int(ch.correct.sum()), len(ch), .5, alternative="greater").pvalue
    R[m].update(Habit_acc=acc(g,"habit_agrees"), Exception_acc=acc(g,"habit_misleads"),
                Gap=gap(g), Gap_CI_low=np.percentile(bs,2.5), Gap_CI_high=np.percentile(bs,97.5),
                Habit_acc_committed=acc(c,"habit_agrees"),
                Exception_acc_committed=acc(c,"habit_misleads"), Gap_committed=gap(c),
                Exception_below_chance_P=pe_lo, Exception_above_chance_P=pe_hi,
                Habit_above_chance_P=ph_hi, Abstention=g.abstained.mean()*100)
    r = R[m]; star = "x" if m in ONESIDED else " "
    print(f"{m:<18}{star}{r['Habit_acc']:>7.1f}{r['Exception_acc']:>8.1f}{r['Gap']:>7.1f}"
          f"{r['Gap_CI_low']:>8.1f} to{r['Gap_CI_high']:>5.1f}  |{r['Habit_acc_committed']:>7.1f}"
          f"{r['Exception_acc_committed']:>9.1f}{r['Gap_committed']:>7.1f}  p<.5={pe_lo:.3f}")
print("  x = excluded as one-sided;  * = committed answers only")

# Holm correction across the informative models for the tests against chance
for col_ in ["Habit_above_chance_P", "Exception_below_chance_P", "Exception_above_chance_P"]:
    for m, v in zip(INFORM, holm([R[m][col_] for m in INFORM])): R[m][col_ + "_Holm"] = v

# median gap with a bootstrap CI that resamples proteins jointly for all models
idx = boot_idx(); P = {m: paired(MAIN[MAIN.model==m]) for m in INFORM}
med_bs = np.median(np.vstack([gap_from(P[m], idx) for m in INFORM]), axis=0)
I = pd.DataFrame({m: R[m] for m in INFORM}).T
print(f"\n  median gap, all {len(MODELS_)}: {np.median([R[m]['Gap'] for m in MODELS_]):.1f} pp"
      f" | informative {len(INFORM)}: {I.Gap.median():.1f} pp "
      f"(95% CI {np.percentile(med_bs,2.5):.1f} to {np.percentile(med_bs,97.5):.1f})")
print(f"  gap positive: all {sum(R[m]['Gap']>0 for m in MODELS_)}/{len(MODELS_)}, "
      f"informative {int((I.Gap>0).sum())}/{len(INFORM)}; per-model CI excludes 0: "
      f"{int((I.Gap_CI_low>0).sum())}/{len(INFORM)}")
print(f"  committed, habit-following: {I.Habit_acc_committed.min():.1f}-{I.Habit_acc_committed.max():.1f}%,"
      f" above chance in {int((I.Habit_above_chance_P_Holm<.05).sum())}/{len(INFORM)} after Holm correction"
      f" ({int((I.Habit_above_chance_P<.05).sum())} uncorrected)")
print(f"  committed, habit-contradicting: {I.Exception_acc_committed.min():.1f}-"
      f"{I.Exception_acc_committed.max():.1f}%; after Holm correction significantly below chance in "
      f"{int((I.Exception_below_chance_P_Holm<.05).sum())}/{len(INFORM)} "
      f"({int((I.Exception_below_chance_P<.05).sum())} uncorrected), above chance in "
      f"{int((I.Exception_above_chance_P_Holm<.05).sum())}/{len(INFORM)}")
print(f"  all responses (abstentions wrong): habit {I.Habit_acc.min():.1f}-{I.Habit_acc.max():.1f}%,"
      f" exception {I.Exception_acc.min():.1f}-{I.Exception_acc.max():.1f}%")
print(f"  median committed gap: {I.Gap_committed.median():.1f} pp; widened vs all-response gap in "
      f"{int((I.Gap_committed>I.Gap).sum())}/{len(INFORM)}")

model                agree mislead    gap         95% CI  | agree* mislead*   gap*  exc* vs 50%
----------------------------------------------------------------------------------------------------
claude-haiku-4.5      44.4    14.8   29.6    18.5 to 40.7  |   87.3     34.8   52.5  p<.5=0.027
deepseek-v3           50.9    13.0   38.0    26.9 to 49.1  |   85.9     23.0   63.0  p<.5=0.000
gemini-3.7-flash      68.5    37.0   31.5    20.4 to 43.5  |   88.1     51.9   36.1  p<.5=0.676
gemma-3-27b           75.0    30.6   44.4    29.6 to 58.3  |   75.0     30.6   44.4  p<.5=0.000


gemma-4-31b           68.5    33.3   35.2    21.3 to 49.1  |   81.3     40.4   40.9  p<.5=0.045
gpt-5.6-luna          65.7    32.4   33.3    20.4 to 45.4  |   86.6     41.2   45.4  p<.5=0.064
gpt-5.6-terra         71.3    43.5   27.8    15.7 to 40.7  |   86.5     50.0   36.5  p<.5=0.541
gpt-oss-120b          50.0    30.6   19.4     7.4 to 31.5  |   90.0     50.8   39.2  p<.5=0.598
llama-3-70b       x   50.5    60.2   -9.7   -26.9 to  7.0  |   50.5     60.2   -9.7  p<.5=0.987


llama-3.1-8b      x   60.2    36.1   24.1     5.6 to 41.7  |   64.4     37.1   27.2  p<.5=0.005
llama-3.3-70b         63.0    33.3   29.6    15.7 to 44.4  |   73.9     42.4   31.6  p<.5=0.096
med42-70b             63.0    44.4   18.5     1.9 to 34.3  |   63.0     44.9   18.1  p<.5=0.167
medgemma-27b          80.6    27.8   52.8    38.9 to 64.8  |   82.1     28.3   53.8  p<.5=0.000
mistral-large         74.1    33.3   40.7    26.9 to 53.7  |   82.5     36.0   46.5  p<.5=0.003
qwen3-235b            51.9    25.9   25.9    13.9 to 38.0  |   83.6     37.3   46.2  p<.5=0.018


qwen3-32b             42.6    16.7   25.9    13.9 to 37.0  |   86.8     37.5   49.3  p<.5=0.056
  x = excluded as one-sided;  * = committed answers only



  median gap, all 16: 29.6 pp | informative 14: 30.6 pp (95% CI 22.7 to 40.3)
  gap positive: all 15/16, informative 14/14; per-model CI excludes 0: 14/14
  committed, habit-following: 63.0-90.0%, above chance in 14/14 after Holm correction (14 uncorrected)
  committed, habit-contradicting: 23.0-51.9%; after Holm correction significantly below chance in 4/14 (7 uncorrected), above chance in 0/14
  all responses (abstentions wrong): habit 42.6-80.6%, exception 13.0-44.4%
  median committed gap: 44.9 pp; widened vs all-response gap in 12/14


In [3]:
# ============================================================================
# 04c — DECLINING TO ANSWER: same protein, habit vs exception question
# ============================================================================
diffs = []
for m in INFORM:
    w = paired(MAIN[MAIN.model==m], "abstained").dropna().astype(bool)
    x = int((~w.habit_agrees & w.habit_misleads).sum()); y = int((w.habit_agrees & ~w.habit_misleads).sum())
    p = binomtest(x, x+y, .5).pvalue if x+y else 1.0
    g = MAIN[MAIN.model==m]
    ab_h = g[g.condition=="habit_agrees"].abstained.mean()*100
    ab_e = g[g.condition=="habit_misleads"].abstained.mean()*100
    R[m].update(Abstention_habit=ab_h, Abstention_exception=ab_e, Abstention_paired_P=p)
    diffs.append(ab_e-ab_h)
    print(f"  {m:<18} declined only the exception {x:>3} | only the habit question {y:>3} | p={p:.3f}")
d = np.round(diffs, 6)
print(f"\n  abstention difference (exception - habit): median {np.median(d):.1f} pp; "
      f"more on exceptions {int((d>0).sum())}, fewer {int((d<0).sum())}, equal {int((d==0).sum())}")
print(f"  any model declining exceptions significantly more: "
      f"{[m for m in INFORM if R[m]['Abstention_paired_P']<.05 and R[m]['Abstention_exception']>R[m]['Abstention_habit']] or 'none'}")

  claude-haiku-4.5   declined only the exception  25 | only the habit question  16 | p=0.211
  deepseek-v3        declined only the exception  21 | only the habit question  19 | p=0.875
  gemini-3.7-flash   declined only the exception  20 | only the habit question  13 | p=0.296
  gemma-3-27b        declined only the exception   0 | only the habit question   0 | p=1.000
  gemma-4-31b        declined only the exception  15 | only the habit question  13 | p=0.851
  gpt-5.6-luna       declined only the exception  13 | only the habit question  16 | p=0.711
  gpt-5.6-terra      declined only the exception   8 | only the habit question  13 | p=0.383
  gpt-oss-120b       declined only the exception  17 | only the habit question  22 | p=0.522
  llama-3.3-70b      declined only the exception  15 | only the habit question   8 | p=0.210
  med42-70b          declined only the exception   1 | only the habit question   0 | p=1.000


  medgemma-27b       declined only the exception   2 | only the habit question   2 | p=1.000
  mistral-large      declined only the exception   6 | only the habit question   9 | p=0.607
  qwen3-235b         declined only the exception  17 | only the habit question  25 | p=0.280
  qwen3-32b          declined only the exception  28 | only the habit question  23 | p=0.576

  abstention difference (exception - habit): median 0.5 pp; more on exceptions 7, fewer 5, equal 2
  any model declining exceptions significantly more: none


In [4]:
# ============================================================================
# 04d — BOTH HABIT DIRECTIONS, and a lean-cancelled gap
# ============================================================================
print(f"{'model':<19}{'gap: habit act.':>16}{'gap: habit inh.':>16}{'balanced':>10}{'says inhibits':>15}")
for m in MODELS_:
    g = MAIN[MAIN.model==m]; c = g[g.committed]
    gs, gi = gap(g[g.protein_usually=="stimulates"]), gap(g[g.protein_usually=="inhibits"])
    R[m].update(Gap_habit_activating=gs, Gap_habit_inhibiting=gi, Gap_balanced=(gs+gi)/2,
                Says_inhibits=(c.pred=="inhibits").mean()*100)
    print(f"{m:<19}{gs:>16.1f}{gi:>16.1f}{(gs+gi)/2:>10.1f}{R[m]['Says_inhibits']:>14.1f}%")
I = pd.DataFrame({m: R[m] for m in INFORM}).T
both = (I.Gap_habit_activating>0) & (I.Gap_habit_inhibiting>0)
rho = spearmanr(I.Says_inhibits, I.Gap_habit_inhibiting-I.Gap_habit_activating)[0]
print(f"\n  n = {MAIN[MAIN.protein_usually=='inhibits'].item_id.nunique()} questions on habit-inhibiting proteins")
print(f"  gap positive in BOTH directions: {int(both.sum())}/{len(INFORM)} (not: {list(I.index[~both])})")
print(f"  gap larger for habit-inhibiting proteins: "
      f"{int((I.Gap_habit_inhibiting>I.Gap_habit_activating).sum())}/{len(INFORM)}")
print(f"  lean-cancelled (balanced) gap positive {int((I.Gap_balanced>0).sum())}/{len(INFORM)}, "
      f"median {I.Gap_balanced.median():.1f} pp")
print(f"  the inhibitor-side excess tracks each model's overall 'inhibits' rate: Spearman rho = {rho:.2f}")

model               gap: habit act. gap: habit inh.  balanced  says inhibits
claude-haiku-4.5               35.7            18.4      27.1          39.6%
deepseek-v3                    31.4            50.0      40.7          52.0%
gemini-3.7-flash               22.9            47.4      35.1          50.9%
gemma-3-27b                    34.3            63.2      48.7          50.0%
gemma-4-31b                    12.9            76.3      44.6          60.6%
gpt-5.6-luna                   25.7            47.4      36.5          52.1%
gpt-5.6-terra                  14.3            52.6      33.5          55.2%
gpt-oss-120b                   18.6            21.1      19.8          51.2%
llama-3-70b                   -68.2            97.4      14.6          89.3%
llama-3.1-8b                   88.6           -94.7      -3.1           2.4%
llama-3.3-70b                   1.4            81.6      41.5          67.8%
med42-70b                     -24.3            97.4      36.5          75.3%

medgemma-27b                   35.7            84.2      60.0          52.8%
mistral-large                  22.9            73.7      48.3          55.3%
qwen3-235b                     20.0            36.8      28.4          47.2%
qwen3-32b                      22.9            31.6      27.2          50.5%

  n = 76 questions on habit-inhibiting proteins
  gap positive in BOTH directions: 13/14 (not: ['med42-70b'])
  gap larger for habit-inhibiting proteins: 13/14
  lean-cancelled (balanced) gap positive 14/14, median 36.5 pp
  the inhibitor-side excess tracks each model's overall 'inhibits' rate: Spearman rho = 0.87


In [5]:
# ============================================================================
# 04e — PAIRED TEST: each protein is its own control (McNemar exact)
# ============================================================================
for m in MODELS_:
    g = MAIN[MAIN.model==m]; p = paired(g)
    ao, bo, pv = mcnemar(p)
    bc = paired(g, "committed").astype("boolean").fillna(False).astype(bool)
    keep = bc.habit_agrees & bc.habit_misleads
    ao_c, bo_c, pv_c = mcnemar(p[keep])
    R[m].update(Discordant_habit_only=ao, Discordant_exception_only=bo, McNemar_P=pv,
                Both_answered_n=int(keep.sum()), Both_answered_habit_only=ao_c,
                Both_answered_exception_only=bo_c, McNemar_P_both_answered=pv_c)
for col_, out in [("McNemar_P","McNemar_P_Holm"), ("McNemar_P_both_answered","McNemar_P_both_answered_Holm")]:
    for m, v in zip(INFORM, holm([R[m][col_] for m in INFORM])): R[m][out] = v
print(f"{'model':<19}{'habit only':>11}{'exc. only':>10}{'p':>10}{'Holm p':>10}  | both answered:"
      f"{'n':>5}{'h':>4}{'e':>4}{'Holm p':>10}")
for m in MODELS_:
    r = R[m]
    print(f"{m:<19}{r['Discordant_habit_only']:>11}{r['Discordant_exception_only']:>10}{r['McNemar_P']:>10.2e}"
          f"{r.get('McNemar_P_Holm', np.nan):>10.2e}  |{'':>14}{r['Both_answered_n']:>5}"
          f"{r['Both_answered_habit_only']:>4}{r['Both_answered_exception_only']:>4}"
          f"{r.get('McNemar_P_both_answered_Holm', np.nan):>10.2e}")
I = pd.DataFrame({m: R[m] for m in INFORM}).T
narrow = (I.Discordant_habit_only/(I.Discordant_habit_only+I.Discordant_exception_only)).idxmin()
print(f"\n  significant (p<0.05): {int((I.McNemar_P<.05).sum())}/{len(INFORM)}, after Holm "
      f"{int((I.McNemar_P_Holm<.05).sum())}/{len(INFORM)}; P from {I.McNemar_P.max():.1e} to {I.McNemar_P.min():.1e}")
print(f"  both questions answered: significant after Holm in "
      f"{int((I.McNemar_P_both_answered_Holm<.05).sum())}/{len(INFORM)}, same direction in "
      f"{int((I.Both_answered_habit_only>I.Both_answered_exception_only).sum())}/{len(INFORM)}")
print(f"  narrowest split: {narrow} ({int(I.loc[narrow,'Discordant_habit_only'])} vs "
      f"{int(I.loc[narrow,'Discordant_exception_only'])})")

model               habit only exc. only         p    Holm p  | both answered:    n   h   e    Holm p
claude-haiku-4.5            40         8  3.31e-06  2.64e-05  |                 30  18   3  4.47e-03
deepseek-v3                 48         7  1.31e-08  1.70e-07  |                 43  31   4  3.81e-05
gemini-3.7-flash            43         9  2.04e-06  2.03e-05  |                 64  26   3  1.37e-04
gemma-3-27b                 67        19  1.94e-07  2.33e-06  |                108  67  19  2.33e-06
gemma-4-31b                 56        18  1.11e-05  7.76e-05  |                 76  44  12  1.67e-04
gpt-5.6-luna                47        11  2.03e-06  2.03e-05  |                 69  36   7  8.96e-05
gpt-5.6-terra               42        12  5.21e-05  3.13e-04  |                 81  36   8  1.78e-04
gpt-oss-120b                36        15  4.60e-03  9.20e-03  |                 43  22   3  7.83e-04
llama-3-70b                 40        50  3.43e-01       nan  |                107  40  50

In [6]:
# ============================================================================
# 04f — THE MODEL'S OWN STATED HABIT
# ----------------------------------------------------------------------------
# Each model was asked what each protein usually does. Proteins where it said
# "not established" drop out of the own-habit split for that model.
# ============================================================================
own_n = db_n = 0
print(f"{'model':<19}{'direction given':>16}{'agree w/ DB':>12}{'DB gap':>8}{'own gap':>9}{'disagree':>9}{'follows own':>12}")
for m in MODELS_:
    u = U[U.model==m].set_index("source").pred
    said = u[u.isin(["stimulates","inhibits"])]
    dbh = U[U.model==m].set_index("source").truth
    g = MAIN[MAIN.model==m].assign(own=lambda d: d.source.map(said)).dropna(subset=["own"])
    owng = (g[g.truth==g.own].correct.mean() - g[g.truth!=g.own].correct.mean())*100
    dis = said.index[said != dbh.reindex(said.index)]
    gd = MAIN[(MAIN.model==m) & MAIN.source.isin(dis) & MAIN.committed]
    fo = int((gd.pred == gd.source.map(said)).sum()); fd = len(gd) - fo
    if m in INFORM: own_n += fo; db_n += fd
    R[m].update(Probe_direction_n=int(len(said)), Habit_agreement=(said==dbh.reindex(said.index)).sum()/108*100,
                Gap_own_habit=owng, Disagreement_follows_own=fo, Disagreement_follows_database=fd)
    print(f"{m:<19}{len(said):>12}/108{R[m]['Habit_agreement']:>11.1f}%{R[m]['Gap']:>8.1f}{owng:>9.1f}"
          f"{len(dis):>9}{(f'{fo/(fo+fd):.0%}' if fo+fd else '-'):>12}")
I = pd.DataFrame({m: R[m] for m in INFORM}).T
dd = I.Gap_own_habit - I.Gap
print(f"\n  agreement with database (of 108): {I.Habit_agreement.min():.1f}-{I.Habit_agreement.max():.1f}%")
print(f"  own-habit gap larger: informative {int((dd>0).sum())}/{len(INFORM)}, all "
      f"{sum(R[m]['Gap_own_habit']>R[m]['Gap'] for m in MODELS_)}/{len(MODELS_)}; increase "
      f"{dd.min():.1f} to {dd.max():.1f} pp; own-habit gaps {I.Gap_own_habit.min():.1f} to {I.Gap_own_habit.max():.1f}")
print(f"  disagreement proteins, {len(INFORM)} models pooled: answers follow the model's own belief "
      f"{own_n} vs database habit {db_n} ({own_n/(own_n+db_n):.0%}); descriptive, models not independent")

model               direction given agree w/ DB  DB gap  own gap disagree follows own
claude-haiku-4.5             94/108       74.1%    29.6     34.0       14         60%
deepseek-v3                  94/108       73.1%    38.0     40.4       15         56%
gemini-3.7-flash            106/108       89.8%    31.5     37.7        9         64%
gemma-3-27b                 108/108       75.0%    44.4     64.8       27         70%
gemma-4-31b                 106/108       76.9%    35.2     49.1       23         72%
gpt-5.6-luna                 99/108       83.3%    33.3     42.4        9         69%
gpt-5.6-terra               102/108       86.1%    27.8     36.3        9         73%
gpt-oss-120b                102/108       84.3%    19.4     27.5       11         79%
llama-3-70b                 107/108       73.1%    -9.7     31.4       28         89%


llama-3.1-8b                108/108       67.6%    24.1     81.5       35         94%
llama-3.3-70b               108/108       81.5%    29.6     50.0       20         84%


med42-70b                   108/108       77.8%    18.5     46.3       24         81%
medgemma-27b                108/108       76.9%    52.8     65.7       25         64%
mistral-large               105/108       76.9%    40.7     51.4       22         62%
qwen3-235b                  102/108       79.6%    25.9     33.3       16         52%
qwen3-32b                    98/108       79.6%    25.9     31.6       12         69%

  agreement with database (of 108): 73.1-89.8%
  own-habit gap larger: informative 14/14, all 16/16; increase 2.5 to 27.8 pp; own-habit gaps 27.5 to 65.7
  disagreement proteins, 14 models pooled: answers follow the model's own belief 265 vs database habit 117 (69%); descriptive, models not independent


In [7]:
# ============================================================================
# 04g — OPTION ORDER, and answer stability between the two runs
# ============================================================================
for m in MODELS_:
    s = SW[SW.model==m]; n = MAIN[(MAIN.model==m) & MAIN.item_id.isin(set(s.item_id))]
    shift = ((s.pred=="inhibits").mean() - (n.pred=="inhibits").mean())*100
    j = n[["item_id","pred"]].merge(s[["item_id","pred"]], on="item_id", suffixes=("_1","_2"))
    R[m].update(Order_shift=shift, Gap_original_subset=gap(n), Gap_reversed=gap(s),
                Gap_order_averaged=(gap(n)+gap(s))/2,
                Same_answer_both_orders=(j.pred_1==j.pred_2).mean()*100,
                Fabricated_refusal=DEC[DEC.model==m].abstained.mean()*100)
I = pd.DataFrame({m: R[m] for m in INFORM}).T
print(I[["Order_shift","Gap_original_subset","Gap_reversed","Gap_order_averaged",
         "Same_answer_both_orders","Fabricated_refusal"]].round(1).to_string())
k = int((I.Gap_reversed>0).sum()); stable = I.Order_shift.abs() < 6
rho, rp = spearmanr(I.Order_shift.abs(), I.Fabricated_refusal)
print(f"\n  reversed order: gap positive {k}/{len(INFORM)}, sign test p = {binomtest(k,len(INFORM),.5).pvalue:.3f}; "
      f"median {I.Gap.median():.1f} -> {I.Gap_reversed.median():.1f} pp; negative: {list(I.index[I.Gap_reversed<0])}")
print(f"  order-averaged gap: median {I.Gap_order_averaged.median():.1f} pp, positive "
      f"{int((I.Gap_order_averaged>0).sum())}/{len(INFORM)}")
print(f"  |shift| >= 20 pts: {list(I.index[I.Order_shift.abs()>=20])}")
print(f"  |shift| < 6 pts ({int(stable.sum())} models): fabricated-pair refusal "
      f"{I[stable].Fabricated_refusal.min():.1f}-{I[stable].Fabricated_refusal.max():.1f}%, reversed gaps "
      f"{I[stable].Gap_reversed.min():.1f}-{I[stable].Gap_reversed.max():.1f}; identical answers in both runs "
      f"{I[stable].Same_answer_both_orders.min():.1f}-{I[stable].Same_answer_both_orders.max():.1f}%")
big = (I[stable].Gap - I[stable].Gap_reversed).abs()
print(f"  largest run-to-run gap change among stable models: {big.max():.1f} pp ({big.idxmax()})")
print(f"  |shift| vs fabricated refusal across models: Spearman rho = {rho:.2f} (p = {rp:.3f})")
print(f"  fabricated-pair refusal, all models: {min(R[m]['Fabricated_refusal'] for m in MODELS_):.1f}-"
      f"{max(R[m]['Fabricated_refusal'] for m in MODELS_):.1f}%")

                  Order_shift  Gap_original_subset  Gap_reversed  Gap_order_averaged  Same_answer_both_orders  Fabricated_refusal
claude-haiku-4.5         -0.9                 29.6          19.4                24.5                     86.1                96.7
deepseek-v3              25.9                 38.0          18.5                28.2                     68.1                90.8
gemini-3.7-flash          5.6                 31.5          27.8                29.6                     87.5                86.7
gemma-3-27b              38.0                 44.4          -9.3                17.6                     62.0                 0.0
gemma-4-31b               1.9                 35.2          28.7                31.9                     91.7                42.5
gpt-5.6-luna              2.3                 33.3          28.7                31.0                     82.9                65.0
gpt-5.6-terra            -1.9                 27.8          31.5                29.6      

In [8]:
# ============================================================================
# 04h — WEB RETRIEVAL and BIOMEDICAL ADAPTATION
# ============================================================================
def gap_diff_ci(d1, d2):
    p1, p2 = paired(d1), paired(d2)
    idx = np.random.default_rng(SEED).integers(0, len(PROTS), (N_BOOT, len(PROTS)))
    bs = gap_from(p1, idx) - gap_from(p2, idx)
    return gap(d1)-gap(d2), np.percentile(bs,2.5), np.percentile(bs,97.5)

print("WEB RETRIEVAL (same model, same questions)")
for m in sorted(WEB.model.unique()):
    w, b = WEB[WEB.model==m], MAIN[MAIN.model==m]
    for cond in ["habit_agrees","habit_misleads"]:
        x = b[b.condition==cond].set_index("item_id").correct
        y = w[w.condition==cond].set_index("item_id").correct.reindex(x.index)
        up, down = int((~x & y).sum()), int((x & ~y).sum())
        print(f"  {m:<17}{cond:<15}{x.mean()*100:5.1f}% -> {y.mean()*100:5.1f}%  fixed {up:>2}, broken {down:>2}, "
              f"McNemar p = {binomtest(up, up+down, .5).pvalue if up+down else 1:.2g}")
    dlt, lo, hi = gap_diff_ci(w, b)
    R[m].update(Gap_with_search=gap(w), Gap_change_search=dlt, Gap_change_search_CI_low=lo,
                Gap_change_search_CI_high=hi)
    print(f"  {m:<17}gap {gap(b):.1f} -> {gap(w):.1f} pp, change {dlt:+.1f} (95% CI {lo:+.1f} to {hi:+.1f})\n")

print("BIOMEDICAL ADAPTATION: MedGemma-27B vs Gemma-3-27B")
for lab_, data in [("original order", MAIN), ("reversed order", SW)]:
    dlt, lo, hi = gap_diff_ci(data[data.model=="medgemma-27b"], data[data.model=="gemma-3-27b"])
    print(f"  {lab_:<15} difference in gap {dlt:+.1f} pp (95% CI {lo:+.1f} to {hi:+.1f})")
for m in ["gemma-3-27b","medgemma-27b"]:
    g = MAIN[MAIN.model==m]
    print(f"  {m:<14} habit {acc(g,'habit_agrees'):.1f}%, exception {acc(g,'habit_misleads'):.1f}%, "
          f"served by {A[A.model==m].model_id.iat[0]} / {A[A.model==m].provider.iat[0]}")
print(f"  Med42-70B vs Llama-3-70B: not interpretable (base model {'llama-3-70b'} is one-sided at "
      f"{skew['llama-3-70b']:.1%}); difference {gap(MAIN,'med42-70b')-gap(MAIN,'llama-3-70b'):+.1f} pp, supplement only")

WEB RETRIEVAL (same model, same questions)
  gemini-3.7-flash habit_agrees    68.5% ->  75.0%  fixed 11, broken  4, McNemar p = 0.12
  gemini-3.7-flash habit_misleads  37.0% ->  55.6%  fixed 22, broken  2, McNemar p = 3.6e-05
  gemini-3.7-flash gap 31.5 -> 19.4 pp, change -12.0 (95% CI -22.2 to -1.9)

  gpt-5.6-terra    habit_agrees    71.3% ->  71.3%  fixed  8, broken  8, McNemar p = 1
  gpt-5.6-terra    habit_misleads  43.5% ->  61.1%  fixed 24, broken  5, McNemar p = 0.00055
  gpt-5.6-terra    gap 27.8 -> 10.2 pp, change -17.6 (95% CI -28.7 to -6.5)

BIOMEDICAL ADAPTATION: MedGemma-27B vs Gemma-3-27B


  original order  difference in gap +8.3 pp (95% CI -2.8 to +19.4)
  reversed order  difference in gap +10.2 pp (95% CI +3.7 to +17.6)
  gemma-3-27b    habit 75.0%, exception 30.6%, served by google/gemma-3-27b-it / DeepInfra
  medgemma-27b   habit 80.6%, exception 27.8%, served by google/medgemma-27b-text-it:featherless-ai / hf
  Med42-70B vs Llama-3-70B: not interpretable (base model llama-3-70b is one-sided at 89.3%); difference +28.2 pp, supplement only


In [9]:
# ============================================================================
# 04i — COLLECTION RECORD: failures, coverage, providers
# ============================================================================
F = A_all[A_all.http!=200].copy()
key = ["model","arm","item_id"]
got = set(map(tuple, A[key].astype(str).values))
F["recovered"] = [tuple(v) in got for v in F[key].astype(str).values]
kind = np.select([F.error.astype(str).str.contains("model blocked"),
                  F.error.astype(str).str.contains("credit|limit", case=False),
                  F.error.astype(str).str.contains("retries exhausted"),
                  F.error.astype(str).str.contains("busy")],
                 ["skipped after a provider refusal (not sent)", "spending-limit error",
                  "timeout / retries exhausted", "model busy"], "other")
print(f"failed attempts: {len(F):,}"); print(pd.Series(kind).value_counts().to_string())
print(f"recovered on a later attempt: {int(F.recovered.sum()):,}; never recovered: {int((~F.recovered).sum())}")
print(F[~F.recovered].groupby(["model","arm"]).size().to_string())

cov = A.pivot_table(index="model", columns="arm", values="item_id", aggfunc="nunique", fill_value=0)
print("\nunique questions answered per model and arm:\n" + cov.to_string())
print(f"\nscored responses {len(A):,}; unparseable {int(A.pred.isna().sum())}; "
      f"successful answers per (model, arm, question): {A.groupby(key).size().value_counts().to_dict()}")

prov = (A.groupby("model").agg(model_id=("model_id","first"), provider=("provider", lambda s: ";".join(sorted(set(s.dropna())))),
                               n_providers=("provider","nunique"), pinned=("pinned", lambda s: s.dropna().iat[0] if s.notna().any() else ""),
                               responses=("item_id","size")).reset_index())
prov["served_via"] = np.where(prov.model_id.str.contains(":featherless-ai"), "Featherless AI via Hugging Face Inference Providers", "OpenRouter")
prov.to_csv(PROC/"providers_observed.csv", index=False)
print("\n" + prov.to_string(index=False))
print(f"\nevery model served by exactly one provider: {bool((prov.n_providers==1).all())}")
for m in MODELS_:
    r = prov.set_index("model").loc[m]; R[m].update(Provider=r.provider, Pinned=r.pinned or "no")

failed attempts: 4,454
skipped after a provider refusal (not sent)    4351
spending-limit error                             89
timeout / retries exhausted                       9
model busy                                        5
recovered on a later attempt: 4,450; never recovered: 4
model        arm    
llama-3-70b  main       2
             swap       1
             usually    1



unique questions answered per model and arm:
arm               main  swap  usually  web
model                                     
claude-haiku-4.5   336   216      108    0
deepseek-v3        336   216      108    0
gemini-3.7-flash   336   216      108  216
gemma-3-27b        336   216      108    0
gemma-4-31b        336   216      108    0
gpt-5.6-luna       336   216      108    0
gpt-5.6-terra      336   216      108  216
gpt-oss-120b       336   216      108    0
llama-3-70b        334   215      107    0
llama-3.1-8b       336   216      108    0
llama-3.3-70b      336   216      108    0
med42-70b          336   216      108    0
medgemma-27b       336   216      108    0
mistral-large      336   216      108    0
qwen3-235b         336   216      108    0
qwen3-32b          336   216      108    0

scored responses 10,988; unparseable 1; successful answers per (model, arm, question): {1: 10988}

           model                                            model_id       provi

In [10]:
# ============================================================================
# 04j — SUPPLEMENTARY TABLES S1, S3, S4, S5 and the numbers file
# ============================================================================
DEV = {"gpt-5.6-terra":"OpenAI","gpt-5.6-luna":"OpenAI","gpt-oss-120b":"OpenAI","gemini-3.7-flash":"Google",
       "gemma-3-27b":"Google","gemma-4-31b":"Google","medgemma-27b":"Google","claude-haiku-4.5":"Anthropic",
       "llama-3-70b":"Meta","llama-3.1-8b":"Meta","llama-3.3-70b":"Meta","mistral-large":"Mistral",
       "qwen3-32b":"Alibaba","qwen3-235b":"Alibaba","deepseek-v3":"DeepSeek","med42-70b":"M42"}
S1 = pd.DataFrame(R).T.infer_objects().rename_axis("Model").reset_index()
S1.insert(1, "Developer", S1.Model.map(DEV))
S1.insert(2, "Excluded_one_sided", np.where(S1.Model.isin(ONESIDED), "yes", ""))
S1.insert(3, "Dominant_label_share", S1.Model.map(lambda m: skew[m]*100))
num = S1.select_dtypes("number").columns
counts = [c for c in num if c.startswith(("Discordant_", "Both_answered_", "Disagreement_")) or c.endswith("_n")]
pcols = [c for c in num if c.endswith("_P") or "_P_" in c]
rest = [c for c in num if c not in pcols + counts]
S1[pcols] = S1[pcols].astype(float).map(lambda v: float(f"{v:.3g}") if pd.notna(v) else v)
S1[rest] = S1[rest].astype(float).round(1)
S1[counts] = S1[counts].astype("Int64")
S1 = S1.sort_values("Gap", ascending=False)
S1.to_csv(SUPP/"tables"/"TableS1_per_model_results.csv", index=False)
print(f"Table S1: {len(S1)} models x {S1.shape[1]} columns")

B = pd.read_json(PROC/"sighabit_v2.jsonl", lines=True)
G = pd.read_csv(PROC/"eligible_proteins.csv", index_col=0)
S3 = B[["item_id","source","target","source_uniprot","target_uniprot","answer","condition","protein_usually",
        "pclass","group","lopsided","n_refs","component","src_stim_share_loo"]].rename(columns={
        "answer":"ground_truth","protein_usually":"protein_habit","pclass":"protein_class",
        "lopsided":"habit_strength","n_refs":"supporting_publications",
        "src_stim_share_loo":"activation_fraction_all_evidence_excluding_item"})
S3.insert(10, "habit_activation_fraction", S3.source.map(G.stim_share).round(4))
S3.to_csv(SUPP/"tables"/"TableS3_benchmark_items.csv", index=False)
print(f"Table S3: {len(S3)} items (habit_activation_fraction = share of >=2-publication partners activated, "
      f"tested interaction included)")
FB = pd.read_json(PROC/"sighabit_v2_full.jsonl", lines=True)
S4 = FB[FB.is_decoy][["item_id","source","target","source_uniprot","target_uniprot"]]
S4.to_csv(SUPP/"tables"/"TableS4_fabricated_pairs.csv", index=False); print(f"Table S4: {len(S4)} fabricated pairs")
S5 = U[["model","source","truth","pred","confidence","rationale"]].rename(
        columns={"truth":"database_habit","pred":"model_stated_habit"})
S5.to_csv(SUPP/"tables"/"TableS5_habit_probes.csv", index=False); print(f"Table S5: {len(S5)} probe responses")

sha = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
       for p in [PROC/"sighabit_v2.jsonl", PROC/"sighabit_v2_full.jsonl", PROC/"sighabit_v2_usually.jsonl", PROC/"prompts_v2.json"]}
out = dict(sha256=sha, n_models=len(MODELS_), n_informative=len(INFORM), one_sided=ONESIDED,
           one_sided_cutoff=ONESIDED_CUTOFF, n_responses=len(A), n_failed_attempts=len(F),
           unparseable=int(A.pred.isna().sum()),
           median_gap_all=round(float(np.median([R[m]["Gap"] for m in MODELS_])), 1),
           median_gap_informative=round(float(I.Gap.median()), 1),
           median_gap_informative_CI=[round(float(np.percentile(med_bs,2.5)),1), round(float(np.percentile(med_bs,97.5)),1)],
           component_items=B.component.value_counts().to_dict())
(SUPP/"supplementary_numbers.json").write_text(json.dumps(out, indent=2) + "\n")
print(json.dumps(out, indent=2))

Table S1: 16 models x 52 columns


Table S3: 216 items (habit_activation_fraction = share of >=2-publication partners activated, tested interaction included)
Table S4: 120 fabricated pairs
Table S5: 1727 probe responses
{
  "sha256": {
    "sighabit_v2.jsonl": "6c04ef4a25e0c52c97b9bcaa0a20f0a61a2e20bd01ac83ae3f71f49f8538bafb",
    "sighabit_v2_full.jsonl": "99be1e9522e4f933e6ff7b6196d0540de2e59f9f51f46e1a6652db9ede06799d",
    "sighabit_v2_usually.jsonl": "941726bab501802565750cb2d5dbdd0e759db4e31e13c5ef48671efba58c2ac5",
    "prompts_v2.json": "8b042ed3a5b15ff61a668dda07b20128dc9a10e5e165834a6ebc83c78e53ac6d"
  },
  "n_models": 16,
  "n_informative": 14,
  "one_sided": [
    "llama-3-70b",
    "llama-3.1-8b"
  ],
  "one_sided_cutoff": 0.85,
  "n_responses": 10988,
  "n_failed_attempts": 4454,
  "unparseable": 1,
  "median_gap_all": 29.6,
  "median_gap_informative": 30.6,
  "median_gap_informative_CI": [
    22.7,
    40.3
  ],
  "component_items": {
    "causal_ppi": 107,
    "collectri": 72,
    "kinaseextra": 37
  }
